In [ ]:
%pip install pycayennelpp requests
import requests
import json
from cayennelpp import LppFrame
import base64
#import pandas
from IPython.display import display, Markdown, Latex
display.max_rows = 4000
display.max_seq_items = 4000
DISPLAY_DOCKLOGS = False

test = "✅""❌"

In [20]:
iot_lora_service = "http://localhost:6041"
url = f"{iot_lora_service}/iot/services"
url_devices = f"{iot_lora_service}/iot/devices"
did = "104868453"
offset = "1"
device_id = f"hsensor{did}_{offset}"

# Set to true if you want add device to lorawan (if wasn't done with postman)
ADD_DEVICE=False


In [21]:
mosquitto_command_template = """
```
mosquitto_pub  -u admin -P password -t v3/sogeiTTN/devices/{deviceid}/up -m '{{
    "app_id": "sogeiTTN",
    "dev_id": "{deviceid}",
    "hardware_serial": "0102030405060708",
    "port": 1,
    "counter": 2,
    "is_retry": false,
    "confirmed": false,
    "payload_raw": "{payload}"
}}'
```
"""

In [22]:
#url = "https://6041-aquarta-bigdataprojecta-oe6lq79ertz.ws-eu114.gitpod.io/iot/services"
#iot_lora_service = "http://iotagent-lora:6041"
ADD_SERVICE = False
if ADD_SERVICE:
  service_dict = {
    "services": [
      {
        "entity_type": "HeightSensor",
        "apikey": "",
        "resource": "70B3D57ED00006F2",
        "cbroker": "http://orion:1026",
        #"type": "Device",

        "static_attributes": [
          {
            "name": "category",
            "type": "Property",
            "value": "sensor"
          },
          {
            "name": "supportedProtocol",
            "type": "Property",
            "value": "ul20"
          },
          {
              "name": "controlledAsset",
              "type": "Relationship"
          }
        ],
        "internal_attributes": {
          "lorawan": {
            "application_server": {
              "host": "mqtt",
              "username": "admin",
              "password": "password",
              "provider": "TTN"
            },
            "app_eui": "70B3D57ED00006F2",
            "application_id": "sogeiTTN",
            "application_key": "BE6996EEE2B2D6AFFD951383C1F3C3BD",
            "data_model": "cayennelpp"
          }
        }
      }
    ]
  }

  service_dict["services"][0].update({
    "attributes": [
          {
              "object_id": "gps_1",
              "name": "location",
              "type": "Property"
          }
        ]
      }
  )
  payload = json.dumps(service_dict)
  headers = {
    'fiware-service': 'openiot',
    'fiware-servicepath': '/',
    'Content-Type': 'application/json'
  }
  #print(payload)
  response = requests.request("POST", url, headers=headers, data=payload)
  print(response.status_code)
  print(response.text)

In [23]:
if ADD_DEVICE:
  # Add device
  payload = json.dumps({
    "devices": [
      {
        "device_id": device_id,
        "entity_name": f"urn:ngsi-ld:Device:hsensor:{device_id}",
        "entity_type": "HeightSensor",
        "internal_attributes": {
        "lorawan": {
              "application_server": {
                  "host": "mqtt",
                  "username": "admin",
                  "password": "password",
                  "provider": "TTN"
              },
              "app_eui": "70B3D57ED00006F2",
              "application_id": "sogeiTTN",
              "application_key": "BE6996EEE2B2D6AFFD951383C1F3C3BD",
              "data_model": "cayennelpp"
          }
        },
        "attributes": [
          {
            "object_id": "gps_1",
            "name": "location",
            "type": "Property"
          }
        ],
        "static_attributes": [
          {
            "name": "controlledAsset",
            "type": "Relationship",
            "object": f"urn:ngsi-ld:Building:waybridge{did}"
          }
        ]
      }
    ]
  })
  headers = {
    'fiware-service': 'openiot',
    'fiware-servicepath': '/',
    'Content-Type': 'application/json'
  }
  print(payload)
  response = requests.request("POST", url_devices, headers=headers, data=payload)
  print(response.status_code)
  print(response.text)


  orion_url = "http://localhost:1026/ngsi-ld/v1/subscriptions/"
  payload = json.dumps({
    "description": "Notify flask of all height changes",
    "name": "Notify_flask_sens_motion",
    "type": "Subscription",
    "entities": [
      {
        "type": "HeightSensor"
      }
    ],
    "notification": {
      "format": "normalized",
      "endpoint": {
        "uri": "http://flaskdash:8000/api/v1/sens_notify",
        "accept": "application/json"
      },
      "showChanges": True
    },
    "@context": "http://context/datamodels.context.jsonld"
  })
  headers = {
    'Fiware-Service': 'openiot',
    'Fiware-ServicePath': '/',
    'NGSILD-Tenant': 'openiot',
    'Content-Type': 'application/ld+json'
  }

  # response = requests.request("POST", orion_url, headers=headers, data=payload)

  # print(response.text)



In [ ]:
frame = LppFrame()
location = (13.1,13.1,13.1)
frame.add_location(1, *location)

# get byte buffer in CayenneLPP format
buffer = bytes(frame)

gfg = base64.b64encode(buffer) 

mosquitto_command = mosquitto_command_template.format(payload=gfg.decode(),deviceid=device_id)
display(Markdown(f'✅  **location {location}** `{mosquitto_command}` '))
!{mosquitto_command}
iota_out = !docker logs  --tail 20 -n bigdata-iota-lora
iota_out = "\n".join(iota_out)
if DISPLAY_DOCKLOGS :
    print(iota_out)
if iota_out.find("Observations sent to CB successfully")>0:
    display(Markdown(f'✅  Sent  '))
else:
    display(Markdown(f'❌  FAIL ❌ '))


In [ ]:
frame = LppFrame()
location = (13.1,13.1,16.1)
frame.add_location(1, *location)

# get byte buffer in CayenneLPP format
buffer = bytes(frame)

gfg = base64.b64encode(buffer) 

mosquitto_command = mosquitto_command_template.format(payload=gfg.decode(),deviceid=device_id)
display(Markdown(f'✅  **location {location}** `{mosquitto_command}` '))
!{mosquitto_command}
iota_out = !docker logs --tail 20 -n bigdata-iota-lora
iota_out = "\n".join(iota_out)
if DISPLAY_DOCKLOGS :
    print(iota_out)
if iota_out.find("Observations sent to CB successfully")>0:
    display(Markdown(f'✅  Sent  '))
else:
    display(Markdown(f'❌  FAIL ❌ '))

In [ ]:
frame = LppFrame()
location = (13.1,13.1,19.1)
frame.add_location(1, *location)

# get byte buffer in CayenneLPP format
buffer = bytes(frame)

gfg = base64.b64encode(buffer) 

mosquitto_command = mosquitto_command_template.format(payload=gfg.decode(),deviceid=device_id)
display(Markdown(f'✅  **location {location}** `{mosquitto_command}` '))
!{mosquitto_command}
iota_out = !docker logs  --tail 20 -n bigdata-iota-lora
iota_out = "\n".join(iota_out)
if DISPLAY_DOCKLOGS :
    print(iota_out)
if iota_out.find("Observations sent to CB successfully")>0:
    display(Markdown(f'✅  Sent  '))
else:
    display(Markdown(f'❌  FAIL ❌ '))

In [27]:
DELETE_SERVICES = False
if DELETE_SERVICES:
  url = f"{iot_lora_service}/iot/services/?resource=70B3D57ED00006F2&apikey="

  payload = {}
  headers = {
    'fiware-service': 'openiot',
    'fiware-servicepath': '/'
  }

  response = requests.request("DELETE", url, headers=headers, data=payload)
  print(response.status_code)
  print(response.text)